# 1-Connexion & Chargement des Données

In [53]:
import pandas as pd
from sqlalchemy import create_engine, text,engine
from dotenv import load_dotenv
import os 
load_dotenv()

# Configuration de la connexion PostgreSQL
# Remplacez les identifiants par les vôtres
DB_USER = os.getenv("DB_USER")
DB_PASSWORD =os.getenv("DB_PASSWORD")
DB_HOST =os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
db_url=f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

engine = create_engine(db_url)

# Chargement du fichier CSV
df=pd.read_csv(r"C:\Users\salli\OneDrive\Desktop\breifs 5\data\financecore_clean.csv")
df

,transaction_id,client_id,date_transaction,montant,devise,taux_change_eur,montant_eur,categorie,produit,agence,...,trimestre,jour_semaine,montant_eur_verifie,ecart_montant,categorie_risque,nb_transactions,montant_moyen,nb_produits,solde_net,taux_rejet
0,TXN000559,CLI0060,2022-04-19 02:31:00,2050.42,EUR,1.00,2050.42,Depot especes,Compte Epargne,Marseille-Vieux-Port,...,2,Tuesday,2050.420000,0.000000,Medium,12,1200.871667,7,21599.14,0.055728
1,TXN001154,CLI0057,2024-06-20 20:51:00,-123.66,GBP,0.86,-143.79,Retrait DAB,Credit Consommation,Unknown,...,2,Thursday,-143.790698,0.000698,High,9,-379.136667,7,6995.71,0.050000
2,TXN000764,CLI0015,2024-08-28 05:03:00,-396.17,EUR,1.00,-396.17,Prelevement,PEA,Lyon-Part-Dieu,...,3,Wednesday,-396.170000,0.000000,Medium,11,-300.231818,5,10808.05,0.040134
3,TXN001598,CLI0045,2024-01-07 08:16:00,225.20,EUR,1.00,225.20,Paiement CB,Credit Consommation,Bordeaux-Meriadeck,...,1,Sunday,225.200000,0.000000,Low,17,-630.922353,7,24235.30,0.045198
4,TXN001873,CLI0034,2024-08-11 19:52:00,935.32,EUR,1.00,935.32,Interets,Credit Immobilier,Bordeaux-Meriadeck,...,3,Sunday,935.320000,0.000000,High,13,26.430769,6,11456.22,0.045198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,TXN001973,CLI0085,2022-07-12 08:13:00,-5814.40,EUR,1.00,-5814.40,Virement international,Livret A,Toulouse-Capitole,...,3,Tuesday,-5814.400000,0.000000,High,16,-724.453125,8,30829.01,0.068063
1996,TXN001769,CLI0040,2024-03-28 11:34:00,296.96,EUR,1.00,296.96,Virement,Credit Immobilier,Unknown,...,1,Thursday,296.960000,0.000000,High,19,434.803684,7,22387.77,0.050000
1997,TXN001738,CLI0106,2022-09-28 04:02:00,-2106.58,EUR,1.00,-2106.58,Virement international,Assurance Vie,Nantes-Commerce,...,3,Wednesday,-2106.580000,0.000000,Medium,16,-257.029375,8,21000.07,0.085000
1998,TXN001210,CLI0098,2024-02-18 12:02:00,-353.74,CHF,0.97,-364.68,Retrait DAB,Compte Epargne,Bordeaux-Meriadeck,...,1,Sunday,-364.680412,0.000412,Medium,15,-866.908000,8,25020.48,0.045198


# 2-Création du Schéma Relationnel PostgreSQL

In [ ]:
with engine.begin() as conn:
    # Suppression des tables si elles existent (pour réinitialisation)
    conn.execute(text("DROP TABLE IF EXISTS transactions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS agences CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS produits CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS segments CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS comptes CASCADE;"))

    # Table Segments
    conn.execute(text("""
        CREATE TABLE segments (
            id_segment SERIAL PRIMARY KEY,
            nom_segment VARCHAR(50) UNIQUE NOT NULL
        );
    """))

    # Table Agences
    conn.execute(text("""
        CREATE TABLE agences (
            id_agence SERIAL PRIMARY KEY,
            nom_agence VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    # Table Produits
    conn.execute(text("""
        CREATE TABLE produits (
            id_produit SERIAL PRIMARY KEY,
            nom_produit VARCHAR(100) UNIQUE NOT NULL
        );
    """))

    # Table Clients
    conn.execute(text("""
        CREATE TABLE clients (
            client_id VARCHAR(50) PRIMARY KEY,
            score_credit INT,
            id_segment INT REFERENCES segments(id_segment)
        );
    """))

 # Table Comptes
    conn.execute(text("""
    CREATE TABLE comptes (
        compte_id SERIAL PRIMARY KEY,
        client_id VARCHAR(50),
        id_agence INT,
        solde NUMERIC(15,2),

        FOREIGN KEY (client_id) REFERENCES clients(client_id),
        FOREIGN KEY (id_agence) REFERENCES agences(id_agence)
    );
    """))

    # Table Transactions
    conn.execute(text("""
        CREATE TABLE transactions (
            transaction_id VARCHAR(50) PRIMARY KEY,
            date_transaction TIMESTAMP,
            montant_eur DECIMAL(15, 2),
            type_operation VARCHAR(50),
            statut VARCHAR(20),
            client_id VARCHAR(50) REFERENCES clients(client_id),
            id_agence INT REFERENCES agences(id_agence),
            id_produit INT REFERENCES produits(id_produit),
            compte_id INT REFERENCES comptes(compte_id)
        );
        
    """))

# 3- Insertion & Normalisation des Données

In [ ]:
# 1. Insertion des tables de référence (Dimensions) 
def insert_dim(table_name, column_csv, column_db):
    # القيم unique من CSV
    unique_values = df[column_csv].dropna().unique()

    # القيم الموجودة في DB
    existing = pd.read_sql(f"SELECT {column_db} FROM {table_name}", engine)
    existing_values = set(existing[column_db])

    # غير القيم الجديدة فقط
    new_values = [v for v in unique_values if v not in existing_values]

    if new_values:
        dim_df = pd.DataFrame({column_db: new_values})
        dim_df.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"{len(new_values)} nouvelles valeurs ajoutées dans {table_name}")
    else:
        print(f"Aucune nouvelle valeur à insérer dans {table_name}")


# Insert dimensions
insert_dim('segments', 'segment_client', 'nom_segment')
insert_dim('agences', 'agence', 'nom_agence')
insert_dim('produits', 'produit', 'nom_produit')

# 2. Récupération des mappings IDs
segments_map = pd.read_sql(
    "SELECT id_segment, nom_segment FROM segments", engine
).set_index('nom_segment')['id_segment'].to_dict()

agences_map = pd.read_sql(
    "SELECT id_agence, nom_agence FROM agences", engine
).set_index('nom_agence')['id_agence'].to_dict()

produits_map = pd.read_sql(
    "SELECT id_produit, nom_produit FROM produits", engine
).set_index('nom_produit')['id_produit'].to_dict()


# 3. Insertion des Clients
clients_df = df[['client_id', 'score_credit_client', 'segment_client']].copy()

# حذف التكرار حسب client_id
clients_df = clients_df.drop_duplicates(subset=['client_id'], keep='first')

# mapping
clients_df['id_segment'] = clients_df['segment_client'].map(segments_map)

clients_final = clients_df[['client_id', 'score_credit_client', 'id_segment']].rename(
    columns={'score_credit_client': 'score_credit'}
)

# حذف clients لي راهوم فـ DB
existing_clients = pd.read_sql("SELECT client_id FROM clients", engine)
existing_ids = set(existing_clients['client_id'])

clients_final = clients_final[~clients_final['client_id'].isin(existing_ids)]

# insertion
if not clients_final.empty:
    clients_final.to_sql('clients', engine, if_exists='append', index=False)
    print(f"{len(clients_final)} clients insérés avec succès.")
else:
    print("Aucun nouveau client à insérer.")


# 4. Insertion des Transactions
tx_df = df.copy()

tx_df['id_agence'] = tx_df['agence'].map(agences_map)
tx_df['id_produit'] = tx_df['produit'].map(produits_map)

tx_final = tx_df[['transaction_id', 'date_transaction', 'montant_eur',
                  'type_operation', 'statut', 'client_id',
                  'id_agence', 'id_produit']]

# حذف التكرار
tx_final = tx_final.drop_duplicates(subset=['transaction_id'])

# حذف transactions لي راهوم فـ DB
existing_tx = pd.read_sql("SELECT transaction_id FROM transactions", engine)
existing_tx_ids = set(existing_tx['transaction_id'])

tx_final = tx_final[~tx_final['transaction_id'].isin(existing_tx_ids)]

# insertion
if not tx_final.empty:
    tx_final.to_sql('transactions', engine, if_exists='append', index=False)
    print("Chargement des transactions terminé avec succès.")
else:
    print("Aucune nouvelle transaction à insérer.")

3 nouvelles valeurs ajoutées dans segments
9 nouvelles valeurs ajoutées dans agences
8 nouvelles valeurs ajoutées dans produits
150 clients insérés avec succès.
Chargement des transactions terminé avec succès.


In [56]:
# 5. Insertion des Comptes

comptes_df = df.groupby(['client_id', 'agence']).agg(
    solde=('montant_eur', 'sum')
).reset_index()

# mapping agence -> id_agence
comptes_df['id_agence'] = comptes_df['agence'].map(agences_map)

comptes_final = comptes_df[['client_id', 'id_agence', 'solde']]

# éviter doublons existants
existing_comptes = pd.read_sql("SELECT client_id, id_agence FROM comptes", engine)

comptes_final = comptes_final.merge(
    existing_comptes,
    on=['client_id', 'id_agence'],
    how='left',
    indicator=True
)

comptes_final = comptes_final[comptes_final['_merge'] == 'left_only']
comptes_final = comptes_final.drop(columns=['_merge'])

# insert
if not comptes_final.empty:
    comptes_final.to_sql('comptes', engine, if_exists='append', index=False)
    print(f"{len(comptes_final)} comptes insérés")
else:
    print("Aucun compte à insérer")

200 comptes insérés


In [62]:
with engine.begin() as conn:
    conn.execute(text("""
    UPDATE transactions t
    SET compte_id = c.compte_id
    FROM comptes c
    WHERE t.client_id = c.client_id;
    """))

# 4-Analyse SQL Avancée & KPIs Métier

In [57]:
# Requête : Moyenne des transactions par agence et par mois
query_kpi = """
SELECT 
    a.nom_agence, 
    EXTRACT(MONTH FROM t.date_transaction) as mois,
    ROUND(AVG(t.montant_eur), 2) as moyenne_mensuelle
FROM transactions t
JOIN agences a ON t.id_agence = a.id_agence
GROUP BY a.nom_agence, mois
ORDER BY a.nom_agence, mois;
"""

result = pd.read_sql(query_kpi, engine)
print(result.head())

# Requête : Clients avec un solde (fictif ici basé sur montant) inférieur à la moyenne
query_sub = """
SELECT client_id, SUM(montant_eur) as solde
FROM transactions
GROUP BY client_id
HAVING SUM(montant_eur) < (SELECT AVG(montant_eur) FROM transactions)
LIMIT 10;
"""
print(pd.read_sql(query_sub, engine))

           nom_agence  mois  moyenne_mensuelle
0  Bordeaux-Meriadeck   1.0             -10.67
1  Bordeaux-Meriadeck   2.0             345.79
2  Bordeaux-Meriadeck   3.0             140.66
3  Bordeaux-Meriadeck   4.0            -122.09
4  Bordeaux-Meriadeck   5.0            -119.53
  client_id     solde
0   CLI0035  -6085.18
1   CLI0028  -5153.97
2   CLI0081  -2781.36
3   CLI0005  -4590.72
4   CLI0120  -2888.86
5   CLI0128  -1546.88
6   CLI0066 -10886.40
7   CLI0007  -8929.98
8   CLI0078  -5444.24
9   CLI0141 -13635.43


In [58]:
with engine.begin() as conn:
    result = conn.execute(text('''
    SELECT 
    s.nom_segment,
    COUNT(*) AS total,

    SUM(CASE 
        WHEN t.statut = 'defaut' THEN 1 
        ELSE 0 
    END) AS nb_defauts,

    ROUND(
        SUM(CASE 
            WHEN t.statut = 'defaut' THEN 1 
            ELSE 0 
        END) * 100.0 / COUNT(*)
    , 2) AS taux_defaut

FROM transactions t
JOIN clients cl ON t.client_id = cl.client_id
JOIN segments s ON cl.id_segment = s.id_segment
JOIN comptes c ON cl.client_id = c.client_id

GROUP BY s.nom_segment;              
'''))
    
for row in result:
    print(row)

('Risque', 516, 0, Decimal('0.00'))
('Standard', 1749, 0, Decimal('0.00'))
('Premium', 443, 0, Decimal('0.00'))


In [64]:
with engine.begin() as conn:
    result = conn.execute(text('''
 SELECT *
FROM transactions t
JOIN comptes c ON t.compte_id = c.compte_id
JOIN clients cl ON c.client_id = cl.client_id
JOIN segments s ON cl.id_segment = s.id_segment
JOIN agences a ON c.id_agence = a.id_agence
JOIN produits p ON t.id_produit = p.id_produit
LIMIT 10;          
'''))
    
for row in result:
    print(row)

('TXN000559', datetime.datetime(2022, 4, 19, 2, 31), Decimal('2050.42'), 'credit', 'complete', 'CLI0060', 1, 1, 82, 82, 'CLI0060', 1, Decimal('13068.03'), 'CLI0060', 645, 1, 1, 'Premium', 1, 'Marseille-Vieux-Port', 1, 'Compte Epargne')
('TXN001154', datetime.datetime(2024, 6, 20, 20, 51), Decimal('-143.79'), 'debit', 'rejete', 'CLI0057', 2, 2, 79, 79, 'CLI0057', 2, Decimal('-143.79'), 'CLI0057', 435, 2, 2, 'Risque', 2, 'Unknown', 2, 'Credit Consommation')
('TXN000764', datetime.datetime(2024, 8, 28, 5, 3), Decimal('-396.17'), 'debit', 'complete', 'CLI0015', 3, 3, 20, 20, 'CLI0015', 3, Decimal('-3619.48'), 'CLI0015', 648, 3, 3, 'Standard', 3, 'Lyon-Part-Dieu', 3, 'PEA')
('TXN001598', datetime.datetime(2024, 1, 7, 8, 16), Decimal('225.20'), 'credit', 'complete', 'CLI0045', 4, 2, 62, 62, 'CLI0045', 4, Decimal('-9009.74'), 'CLI0045', 704, 3, 3, 'Standard', 4, 'Bordeaux-Meriadeck', 2, 'Credit Consommation')
('TXN001873', datetime.datetime(2024, 8, 11, 19, 52), Decimal('935.32'), 'credit', '

# Créer des vues analytiques pour les KPIs du dashboard 


In [66]:
with engine.begin() as conn:
    result = conn.execute(text('''
CREATE OR REPLACE VIEW v_kpi_global_transactions AS
SELECT
    COUNT(*) AS total_transactions,
    SUM(montant_eur) AS montant_total,
    AVG(montant_eur) AS montant_moyen
FROM transactions;            
'''))

In [67]:
with engine.begin() as conn:
    result = conn.execute(text('''
CREATE OR REPLACE VIEW v_kpi_statut_transactions AS
SELECT
    statut,
    COUNT(*) AS nb_transactions,
    SUM(montant_eur) AS montant_total
FROM transactions
GROUP BY statut;            
'''))